In [2]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical
import numpy as np 
# 데이터 로드 및 전처리
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# 평균 및 표준편차 계산
mean = np.mean(x_train, axis=(0, 1, 2, 3))
std = np.std(x_train, axis=(0, 1, 2, 3))

# 정규화
x_train = (x_train - mean) / (std + 1e-7) # 분모가 0이 되는 것을 방지하기 위해 작은 값 추가
x_test = (x_test - mean) / (std + 1e-7)

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)
# 모델 정의
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])

# 컴파일 및 학습
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()
model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 30, 30, 32)        896       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 15, 15, 32)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 13, 13, 64)        18496     
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 6, 6, 64)         0         
 2D)                                                             
                                                                 
 flatten (Flatten)           (None, 2304)              0         
                                                                 
 dense (Dense)               (None, 128)               2

In [3]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam

# 1. 데이터 불러오기
(ds_train, ds_test), info = tfds.load(
    'mnist',
    split=['train', 'test'],
    as_supervised=True,
    with_info=True
)

# 2. 데이터 정규화 및 배치 처리 함수
def preprocess(image, label):
    image = tf.image.convert_image_dtype(image, tf.float32)
    image = tf.reshape(image, (28, 28, 1))  # 명시적으로 채널 차원 추가
    return image, label

# 데이터 전처리 및 배치 적용
BATCH_SIZE = 32
ds_train = ds_train.map(preprocess).cache().shuffle(info.splits['train'].num_examples).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_test.map(preprocess).batch(BATCH_SIZE).cache().prefetch(tf.data.AUTOTUNE)

# 3. 모델 만들기
model = Sequential([
    Flatten(input_shape=(28, 28, 1)),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])

# 4. 모델 컴파일
model.compile(
    optimizer=Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 5. 모델 학습
model.fit(
    ds_train,
    epochs=5,
    validation_data=ds_test
)

# 6. 모델 평가
loss, accuracy = model.evaluate(ds_test)
print("테스트 정확도:", accuracy)

Epoch 1/5
1875/1875 [==============================] - 12s 4ms/step - loss: 0.2595 - accuracy: 0.9263 - val_loss: 0.1312 - val_accuracy: 0.9619
Epoch 2/5
1875/1875 [==============================] - 8s 4ms/step - loss: 0.1116 - accuracy: 0.9673 - val_loss: 0.0989 - val_accuracy: 0.9696
Epoch 3/5
1875/1875 [==============================] - 8s 4ms/step - loss: 0.0763 - accuracy: 0.9773 - val_loss: 0.0856 - val_accuracy: 0.9743
Epoch 4/5
1875/1875 [==============================] - 8s 4ms/step - loss: 0.0566 - accuracy: 0.9831 - val_loss: 0.0701 - val_accuracy: 0.9776
Epoch 5/5
313/313 [==============================] - 1s 3ms/step - loss: 0.0815 - accuracy: 0.9748
테스트 정확도: 0.9747999906539917
